# Energia erosiva de la lluvia
Este simulador es, en esencia, un **laboratorio virtual de física de suelos**. Su objetivo no es solo decir "la vegetación es buena", sino cuantificar el impacto de las gotas de lluvia y la superficie terrestre a través de tres escenarios distintos: **Bosque**, **Pastizal** y **Suelo Desnudo**.

### Los Pilares del Modelo

El sistema se basa en la interacción de dos fenómenos contrapuestos:

1. **La Intercepción (El efecto "paraguas"):** Calcula cuánta agua se queda atrapada en el dosel (las hojas) y nunca llega a tocar el suelo.

2. **La Energía Cinética (El efecto "proyectil"):** Calcula la fuerza del impacto. Aquí el modelo distingue dos fuentes de energía: Lluvia directa (direct throughfall) y Goteo de las hojas (leaf drainage)

### El Objetivo Final: El Índice de Impacto

El modelo multiplica el **volumen neto de agua** que logra atravesar la vegetación por la **fuerza (energía)** de cada milímetro de esa agua. El resultado es la **energia total**, un número que representa la capacidad de esa tormenta para romper los agregados del suelo.

Al mover los controles, lo que realmente estás haciendo es cambiar la **arquitectura del paisaje** (la altura de los árboles) y la **magnitud del evento climático** (la lluvia total), para observar cómo estas piezas encajan en la protección —o vulnerabilidad— de un ecosistema mediterráneo.

In [25]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

def compare_protection_models(rain_mm, forest_h):
    # --- CONFIGURACIÓN DE TAMAÑOS DE FUENTE ---
    plt.rcParams.update({'font.size': 14}) # Tamaño base para todo
    fs_title = 20
    fs_label = 18
    fs_ticks = 16
    fs_data = 16

    # --- 1. Parámetros ---
    I_mediterraneo = 30.0  
    forest_lai, a_forest = 5.0, 0.5
    grass_lai, grass_h, a_grass = 2.0, 0.25, 0.5

    # --- 2. Intercepción (Braden) ---
    def get_interception(a, LAI, r_mm):
        if LAI == 0: return 0
        r_cm = r_mm / 10.0
        pi_cm = a * LAI * (1 - (1 / (1 + (r_cm / (3 * a)))))
        return pi_cm * 10.0

    int_forest = get_interception(a_forest, forest_lai, rain_mm)
    int_grass = get_interception(a_grass, grass_lai, rain_mm)
    
    through_forest = max(0, rain_mm - int_forest)
    through_grass = max(0, rain_mm - int_grass)

    # --- 3. Energía Cinética Unitara ---
    def get_ke_veg(h):
        if h < 0.15: return 0
        h_eff = min(h, 8)
        return 15.8 * (h_eff**0.5) - 5.87

    ke_unit_forest = get_ke_veg(forest_h)
    ke_unit_grass = get_ke_veg(grass_h)
    ke_unit_bare = 11.9 + 8.73 * np.log10(I_mediterraneo)

    # --- 4. Impacto Erosivo Total ---
    impact_forest = through_forest * ke_unit_forest
    impact_grass = through_grass * ke_unit_grass
    impact_bare = rain_mm * ke_unit_bare

    # --- 5. Visualización Mejorada ---
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(22, 8))
    labels = ['Bosque', 'Pastizal', 'Suelo Desnudo']
    colors = ['#1b5e20', '#4caf50', '#8d6e63']
    
    # Plot 1: Intercepción
    ax1.bar(labels, [int_forest, int_grass, 0], color=colors)
    ax1.set_title(f'Agua Retenida (mm)\n(Lluvia total: {rain_mm}mm)', fontsize=fs_title, fontweight='bold', pad=20)
    ax1.set_ylabel('mm', fontsize=fs_label)
    ax1.tick_params(axis='both', labelsize=fs_ticks)
    ax1.set_ylim(0, 25)

    # Plot 2: Energía Cinética por mm
    ax2.bar(labels, [ke_unit_forest, ke_unit_grass, ke_unit_bare], color=['#d32f2f', '#ff9800', '#f44336'])
    ax2.set_title(f'Energía por mm\n(I = {I_mediterraneo} mm/h)', fontsize=fs_title, fontweight='bold', pad=20)
    ax2.set_ylabel('$J/m^2$ por mm', fontsize=fs_label)
    ax2.tick_params(axis='both', labelsize=fs_ticks)
    ax2.set_ylim(0, 50)

    # Plot 3: Impacto Total
    impacts = [impact_forest, impact_grass, impact_bare]
    bars = ax3.bar(labels, impacts, color=['#212121', '#757575', '#5d4037'])
    ax3.set_title('Energia Total', fontsize=fs_title, fontweight='bold', pad=20)
    ax3.set_ylabel('$J/m^2$', fontsize=fs_label)
    ax3.tick_params(axis='both', labelsize=fs_ticks)
    
    # Etiquetas de datos sobre las barras
    for bar in bars:
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + (max(impacts)*0.02),
                 f'{height:.1f}', ha='center', va='bottom', 
                 fontweight='bold', fontsize=fs_data)
    
    ax3.set_ylim(0, max(impacts) * 1.25 if max(impacts) > 0 else 500)
    
    plt.tight_layout(pad=3.0)
    plt.show()

interact(compare_protection_models, 
         rain_mm=IntSlider(value=1, min=1, max=60, step=1, description='Lluvia (mm):'),
         forest_h=FloatSlider(value=5, min=0, max=10, step=0.5, description='Alt. Árbol (m):'))

interactive(children=(IntSlider(value=1, description='Lluvia (mm):', max=60, min=1), FloatSlider(value=5.0, de…

<function __main__.compare_protection_models(rain_mm, forest_h)>

## Preguntas
### 1. El Reto de la Intercepción

* Deja la altura del árbol fija y aumenta la lluvia poco a poco. ¿A partir de cuántos milímetros de lluvia notas que las barras de "Agua Retenida" del Bosque y el Pastizal dejan de crecer significativamente? ¿por qué?

* Con una lluvia de 10 mm, ¿qué porcentaje del agua logra detener el bosque frente al pastizal? ¿Y con 60 mm? ¿por qué?

### 2. La "Paradoja del Bosque"

* ¿A qué altura la energía de un mm de lluvia del bosque iguala a la del suelo desnudo? ¿tiene sentido? ¿por que?

* ¿Existe alguna altura del árbol en la que el bosque logre tener *menos* energía por mm que el pastizal?

### 3. El Escenario Crítico

* Mantén la lluvia al máximo (60 mm). ¿Cuántas veces más erosivo es el Suelo Desnudo comparado con el Pastizal?

### 4. La mejor cubierta

* ¿cual es mejor para proteger el suelo de la erosion, el bosque o el pastizal? ¿que caracteristica le hace tan buen protector, su capacidad de retener el agua de lluvia o su altura?

* ¿es la vegetacion siempre buena para proteger el suelo de la erosion de la lluvia?

### 5. Representacion del Bosque en el modelo

* ¿la representacion del bosque en este modelo es realista? ¿falta algo importante en dicha representacion (es decir algo que normalmente vemos en la naturaleza pero que el modelo no tiene en cuenta) y que puede cambiar su efecto protector?
